# 01 , Exploration des données brutes

**Projet Data Science , Prédiction des retards de vols aériens**

Objectif de ce notebook : comprendre le jeu de données *avant* de coder quoi que ce soit , 
quelles colonnes, quelle **cible** prédire, quels **pièges** (fuite de données, valeurs manquantes).
Ces décisions justifient le schéma de la base et les features utilisées ensuite.


## 1. La problématique

On veut prédire si un vol arrivera **en retard** (≥ 15 min), *avant* qu'il ne parte, à partir
d'informations connues à la réservation : compagnie, aéroports, horaire prévu, jour, distance.
C'est un vrai problème d'apprentissage : le retard est **incertain** (météo, congestion, rotation
des avions...), contrairement par ex. à une empreinte carbone qui est une simple formule.

## 2. La source des données

- **US Bureau of Transportation Statistics (BTS)** , *On-Time Performance* : tous les vols
 intérieurs américains, un fichier par mois. Source : <https://www.transtats.bts.gov/>
- On télécharge l'**année 2024 complète** (12 mois) ≈ **7 millions de vols** vers volume "Big Data".
- Coordonnées des aéroports (pour la carte) : **OpenFlights** <https://openflights.org/data.html>

Le téléchargement est automatisé dans `src/extract.py`. Regardons d'abord **un seul mois** brut.


In [1]:
import duckdb, pandas as pd
pd.set_option("display.max_columns", 40)
raw = "../data/raw/bts_2024_01.csv"
con = duckdb.connect()
n = con.execute(f"SELECT count(*) FROM read_csv_auto('{raw}')").fetchone()[0]
cols = [c[0] for c in con.execute(f"SELECT * FROM read_csv_auto('{raw}') LIMIT 0").description]
print("Janvier 2024 :", f"{n:,}", "vols et", len(cols), "colonnes")

Janvier 2024 : 547,271 vols et 110 colonnes


**110 colonnes** pour un seul mois : beaucoup sont inutiles pour nous (données de déroutement
`Div1..Div5`, identifiants techniques...). On va garder l'essentiel. Voici les 30 premières :

In [2]:
cols[:30]

['Year',
 'Quarter',
 'Month',
 'DayofMonth',
 'DayOfWeek',
 'FlightDate',
 'Reporting_Airline',
 'DOT_ID_Reporting_Airline',
 'IATA_CODE_Reporting_Airline',
 'Tail_Number',
 'Flight_Number_Reporting_Airline',
 'OriginAirportID',
 'OriginAirportSeqID',
 'OriginCityMarketID',
 'Origin',
 'OriginCityName',
 'OriginState',
 'OriginStateFips',
 'OriginStateName',
 'OriginWac',
 'DestAirportID',
 'DestAirportSeqID',
 'DestCityMarketID',
 'Dest',
 'DestCityName',
 'DestState',
 'DestStateFips',
 'DestStateName',
 'DestWac',
 'CRSDepTime']

## 3. Quelle cible ?

BTS fournit `ArrDel15` = 1 si le vol arrive **au moins 15 minutes en retard** (définition officielle
BTS du retard). C'est notre cible de classification. On regarde aussi les taux d'annulation/déroutement.

In [3]:
con.execute(f'''SELECT round(avg(ArrDel15),3) AS taux_retard_arrivee,
 round(avg(DepDel15),3) AS taux_retard_depart,
 round(avg(Cancelled),4) AS taux_annulation,
 round(avg(Diverted),4) AS taux_deroutement
 FROM read_csv_auto('{raw}')''').df()

,taux_retard_arrivee,taux_retard_depart,taux_annulation,taux_deroutement
0,0.241,0.232,0.0373,0.0028


≈ **24 %** des vols sont en retard à l'arrivée vers classes déséquilibrées (76/24). On évaluera donc
avec l'AUC et le F1, pas seulement l'accuracy (un modèle qui dit "jamais en retard" ferait déjà 76 %).

## 4. Le piège : la fuite de données (*data leakage*)

Une erreur classique : utiliser une information qu'on **ne connaît pas encore** au moment de la
prédiction. Ici, le retard au **départ** (`DepDelay`) n'est connu qu'une fois l'avion parti.
Regardons sa corrélation avec le retard à l'arrivée.
*(Réf. pièges courants scikit-learn : <https://scikit-learn.org/stable/common_pitfalls.html>)*

In [4]:
corr = con.execute(f"SELECT corr(DepDelay, ArrDelay) FROM read_csv_auto('{raw}')").fetchone()[0]
print("corr(DepDelay, ArrDelay) =", round(corr,3))

corr(DepDelay, ArrDelay) = 0.971


**0,97** : si on gardait `DepDelay`, prédire l'arrivée deviendrait trivial (on tricherait).
Idem pour les colonnes de *causes* (`CarrierDelay`, `WeatherDelay`...) : elles ne sont remplies
**que** lorsqu'il y a déjà un retard. Vérifions leur taux de valeurs manquantes.

In [5]:
con.execute(f'''SELECT
 round(avg(CASE WHEN CarrierDelay IS NULL THEN 1 ELSE 0 END),3) AS null_carrier,
 round(avg(CASE WHEN WeatherDelay IS NULL THEN 1 ELSE 0 END),3) AS null_weather,
 round(avg(CASE WHEN ArrDelay IS NULL THEN 1 ELSE 0 END),3) AS null_arrdelay
 FROM read_csv_auto('{raw}')''').df()

,null_carrier,null_weather,null_arrdelay
0,0.769,0.769,0.04


≈ **77 % de valeurs nulles** sur les causes (= la proportion de vols à l'heure) vers ces colonnes
fuient la cible ET sont majoritairement vides. On les **écarte**. Les 4 % de `ArrDelay` manquants
correspondent aux vols annulés/déroutés (pas d'arrivée) vers on les retirera pour la cible.

## 5. Décision : les features retenues (connues *avant* le vol)

| Feature | Type | Pourquoi |
|---|---|---|
| `airline` | catégorielle | certaines compagnies sont plus ponctuelles |
| `origin`, `dest` | catégorielles | congestion propre à chaque aéroport |
| `dep_hour` | numérique | les retards s'accumulent en fin de journée |
| `day_of_week`, `month` | numériques | week-ends, saisons (météo/vacances) |
| `distance`, `crs_elapsed` | numériques | durée prévue du vol |

Ces choix sont implémentés en SQL dans `src/transform.py`, qui charge tout dans **DuckDB**.
Voici le résultat : la table propre `flights`.

In [6]:
db = duckdb.connect("../data/processed/flights.duckdb", read_only=True)
db.execute("SELECT flight_date, airline, origin, dest, dep_hour, distance, arr_del15 FROM flights LIMIT 5").df()

,flight_date,airline,origin,dest,dep_hour,distance,arr_del15
0,2024-08-28,UA,LAS,IAH,10,1222,0
1,2024-02-20,UA,PHX,EWR,7,2133,0
2,2024-04-27,UA,SAT,EWR,7,1569,0
3,2024-04-27,UA,EWR,PBI,13,1023,0
4,2024-04-27,UA,PBI,EWR,17,1023,0


In [7]:
print("Total après nettoyage :", f"{db.execute('SELECT count(*) FROM flights').fetchone()[0]:,}", "vols")
db.execute("DESCRIBE flights").df()

Total après nettoyage : 7,079,061 vols


,column_name,column_type,null,key,default,extra
0,flight_date,DATE,YES,None,None,None
1,month,BIGINT,YES,None,None,None
2,day_of_week,BIGINT,YES,None,None,None
3,dep_hour,INTEGER,YES,None,None,None
4,dep_min,INTEGER,YES,None,None,None
5,dep_block,VARCHAR,YES,None,None,None
6,airline,VARCHAR,YES,None,None,None
7,tail,VARCHAR,YES,None,None,None
8,origin,VARCHAR,YES,None,None,None
9,dest,VARCHAR,YES,None,None,None


**Bilan de l'exploration** : ~7 M de vols, cible `arr_del15` (~24 % de positifs), features
uniquement pré-vol pour éviter la fuite de données. La suite : l'**analyse exploratoire (EDA)**
dans le notebook `02`, puis la **modélisation** dans le `03`.